# Imports & Parameters

In [1]:
import os
import re
import time

import numpy as np
import pandas as pd
import hiplot as hip
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import accuracy_score, classification_report, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)
os.environ['PYDEVD_DISABLE_FILE_VALIDATION'] = '1'


## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

In [2]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

data_dumper_path = '/Users/rapido/local-datasets/customer/appography/appography_data_dump/'
local_database = '/Users/rapido/local-datasets/customer/appography/appography_v2/bangalore/database/'

/Users/rapido/analytics/customer/Appography/appography_demographic_dataset


In [3]:
##Parameter 
start_date = '20240301'
end_date = '20240331'
city = 'Bangalore'

city_service = f"""
    select 
        service_level city_service
    from 
        datasets.service_mapping 
    where 
        city_display_name = '{city}' 
        and service_type != 'delivery'  
        and service_category != 'delivery'
"""
df_city_service = pd.read_sql(city_service, connection)
service = ', '.join("'" + value + "'" for value in sorted(df_city_service['city_service']))

In [4]:
print("\033[1m Date Range : \033[0m", start_date, " to ", end_date)
print("\033[1m City : \033[0m", city)
print("\033[1m Service : \033[0m", service)

 Date Range :  20240301  to  20240331
 City :  Bangalore
 Service :  'Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite'


# Datasets

In [5]:
# Get the current working directory
cwd = os.getcwd()
print(cwd)

/Users/rapido/analytics/customer/Appography/appography_demographic_dataset


## Customer base

### Metric data

print('Query 1 running..')
start_time = time.time()

active_customer_base = f"""

    WITH fe_tbl AS (
    
        SELECT 
            yyyymmdd,
            platform,
            service_detail_id,
            user_id customer_id,
            fare_estimate_id,
            quarter_hour fe_quarter_hour
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')
        ),
        
        rr_net_tbl as (
        
        SELECT   
            yyyymmdd,
            service_detail_id,
            customer_id,
            estimate_id AS fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag,
            quarter_hour rr_quarter_hour
        FROM 
            orders.order_logs_snapshot ols
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city_name = '{city}'
            AND service_obj_service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite')
        ),
        
        merged AS (
        
        SELECT
            fe_tbl.*,
            rr_net_tbl.order_id,
            rr_net_tbl.order_status,
            rr_net_tbl.spd_fraud_flag,
            rr_net_tbl.rr_quarter_hour
        FROM
            fe_tbl
        LEFT JOIN
            rr_net_tbl
            ON fe_tbl.yyyymmdd = rr_net_tbl.yyyymmdd
            AND fe_tbl.customer_id = rr_net_tbl.customer_id
            AND fe_tbl.service_detail_id = rr_net_tbl.service_detail_id
        )

        SELECT
            platform,
            customer_id,
            COUNT(DISTINCT fare_estimate_id) AS fe_count,
            COUNT(DISTINCT order_id) AS rr_count,
            COUNT(DISTINCT CASE WHEN order_status = 'dropped' AND spd_fraud_flag != True THEN order_id END) AS net_count,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN fare_estimate_id END) morning_fe,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(fe_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN fare_estimate_id END) evening_fe,

            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 8 AND 12 THEN order_id END) morning_rr,
            COUNT(DISTINCT CASE WHEN CAST(SUBSTR(rr_quarter_hour,1,2) AS REAL) BETWEEN 13 AND 22 THEN order_id END) evening_rr

        FROM 
            merged
        GROUP BY 1,2
        
"""
df_active_customer_base = pd.read_sql(active_customer_base, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_active_customer_base.head(1)

In [6]:
# df_active_customer_base.to_csv(local_database + 'raw_active_customer_base.csv', index=False)
df_active_customer_base = pd.read_csv(local_database + 'raw_active_customer_base.csv')

### Segment data

print('Query 2 running..')
start_time = time.time()
customer_segments = f"""

     SELECT
        customer_id,
        taxi_ltr_segment ltr_segment,
        taxi_lifetime_rides life_time_rides,
        taxi_lifetime_stage life_time_stage,
        taxi_service_affinity service_affinity

    FROM 
        datasets.iallocator_customer_segments
    WHERE 
        run_date =  DATE_FORMAT(DATE_PARSE('{end_date}', '%Y%m%d') + INTERVAL '1' DAY , '%Y-%m-%d')
        AND taxi_recency_segment != 'INACTIVE' 
        
"""
df_customer_segments = pd.read_sql(customer_segments, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_segments.head(1)

In [7]:
# df_customer_segments.to_csv(local_database + 'raw_customer_segments.csv', index=False)
df_customer_segments = pd.read_csv(local_database + 'raw_customer_segments.csv')

### Age and gender data

print('Query 3 running..')
start_time = time.time()

customer_demo_data = f"""

        WITH fe_customer AS (

        SELECT 
            user_id customer_id
        FROM 
            pricing.fare_estimates_enriched 
        WHERE 
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND city = '{city}'
            AND service_name IN ('Auto', 'Auto Pool', 'Bike Lite', 'CabEconomy', 'CityAuto', 'Link', 'Lite')
            AND platform IN ('android', 'ios')
        GROUP BY 1
        ),

        demographic_data AS ( 

        SELECT
            customerId customer_id,
            CASE WHEN age BETWEEN 15 AND 60 THEN age ELSE 0 END age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            datasets.customer_single_view
        )

        SELECT
            a.customer_id,
            COALESCE(age, 0) age,
            COALESCE(gender, 'UNKNOWN') gender
        FROM 
            fe_customer a
        LEFT JOIN 
            demographic_data b
            on a.customer_id = b.customer_id        
"""

df_customer_demo_data = pd.read_sql(customer_demo_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_demo_data.head(1)

In [8]:
# df_customer_demo_data.to_csv(local_database + 'raw_customer_demo_data.csv', index=False)
df_customer_demo_data = pd.read_csv(local_database + 'raw_customer_demo_data.csv')

### Use-case data

print('Query 4 running..')
start_time = time.time()

customer_use_case_data = f"""

    WITH use_case AS (

        SELECT
            hex_10, 
            usecase_tag
        FROM
            (
            SELECT 
                hex_10, 
                combined_final_usecase_accuracy as usecase_tag,
                ROW_NUMBER() OVER(PARTITION BY hex_10 ORDER BY run_date DESC) seq_no
            FROM
                hive.experiments_internal.combined_usecase_hex_10_level
            WHERE 
                aoi = 'Bangalore, India'
            )
        WHERE   
            seq_no = 1
        ),

        active_customers AS (

        SELECT 
            customer_id,
            order_id,
            drop_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'first_half_drop' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON drop_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 8 
            AND cast(substr(quarter_hour,1,2) as real) <= 12    
            AND city_name = '{city}'
            
        GROUP BY 1,2,3,4

        UNION 

            SELECT 
            customer_id,
            order_id,
            pickup_location_hex_10 hex_10,
            COALESCE(usecase_tag, 'Unknown') usecase_tag,
            'second_half_pickup' time_period
        FROM 
            orders.order_logs_snapshot
        LEFT JOIN 
            use_case a 
            ON pickup_location_hex_10 = hex_10
        WHERE
            yyyymmdd BETWEEN '{start_date}' AND '{end_date}'
            AND channel_host = 'android'
            AND cast(substr(quarter_hour,1,2) as real) >= 13
            AND cast(substr(quarter_hour,1,2) as real) <= 21
            AND city_name = '{city}'
        GROUP BY 1,2,3,4

        ),

        merge AS (

        SELECT
            customer_id,
            time_period,
            usecase_tag,
            orders,
            ROW_NUMBER() OVER(PARTITION BY customer_id,time_period ORDER BY orders DESC) seq_no
        FROM 
            (
            SELECT
                customer_id,
                time_period,
                usecase_tag,
                COUNT(DISTINCT order_id) orders
            FROM 
                active_customers
            GROUP BY 1,2,3
            )
        -- WHERE 
        --     LOWER(usecase_tag) != 'unknown'
        )

        SELECT
            customer_id,
            COALESCE(first_half_drop_primary, 'NA') first_half_drop_primary,
            COALESCE(first_half_drop_secondary, 'NA') first_half_drop_secondary,
            COALESCE(second_half_pickup_primary, 'NA') second_half_pickup_primary,
            COALESCE(second_half_pickup_secondary, 'NA') second_half_pickup_secondary
        FROM
        (
        SELECT
            customer_id,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 1 THEN usecase_tag END) first_half_drop_primary,
            MAX(CASE WHEN time_period = 'first_half_drop' AND seq_no = 2 THEN usecase_tag END) first_half_drop_secondary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 1 THEN usecase_tag END) second_half_pickup_primary,
            MAX(CASE WHEN time_period = 'second_half_pickup' AND seq_no = 2 THEN usecase_tag END) second_half_pickup_secondary
        FROM 
            merge 
        WHERE 
            seq_no in (1,2)
        GROUP BY 1
        ) 
"""

df_customer_use_case_data = pd.read_sql(customer_use_case_data, connection)
end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_customer_use_case_data.head(1)

In [9]:
# df_customer_use_case_data.to_csv(local_database + 'raw_customer_use_case_data.csv', index=False)
df_customer_use_case_data = pd.read_csv(local_database + 'raw_customer_use_case_data.csv')

### Appography data

print('Query 5 running..')
start_time = time.time()

date_range = pd.date_range(start=start_date, end=end_date, freq='D').strftime('%Y%m%d')
file_names = [f"users_app_list_{date}.csv" for date in date_range]
csv_files = [file for file in os.listdir(data_dumper_path) if file.endswith('.csv') and file in file_names]
#csv_files = ['users_app_list_20240325.csv'] # testing...

dfs = []

#Loop through each CSV file, read it into a DataFrame, and append to the list
for file in csv_files:
    file_path = os.path.join(data_dumper_path, file)
    df = pd.read_csv(file_path)
    dfs.append(df)

#Concatenate all DataFrames in the list into a single DataFrame
combined_data_dumper = pd.concat(dfs, ignore_index=True)

print("Fetched and Concatenated all files")

#Only selected city
df_combined_data_dumper = pd.merge(df_customer_demo_data['customer_id'],
                                   combined_data_dumper,
                                   how='inner',
                                   left_on=['customer_id'],
                                   right_on=['user_id'] 
                                  ) 
print("Merged")

#delete combined_data_dumper
del combined_data_dumper

#Latest updated data
df_combined_data_dumper['last_sync_date'] = df_combined_data_dumper['last_sync_date'].astype('int32')
df_combined_data_dumper = df_combined_data_dumper.sort_values(by=['user_id', 'last_sync_date'], ascending=[True, False])
df_combined_data_dumper = df_combined_data_dumper.drop_duplicates(subset='user_id', keep='first')
df_combined_data_dumper = df_combined_data_dumper[['customer_id', 'app_list_set']]

print("Updated data")

#regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_combined_data_dumper['app_list_set'] = df_combined_data_dumper['app_list_set'].apply(lambda x: extract_and_trim(x))
df_customer_appography_data = df_combined_data_dumper[['customer_id', 'app_list_set']]

print("regular-exp")

#delete df_combined_data_dumper
del df_combined_data_dumper

end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time : ", execution_time, " mins")

In [10]:
# df_customer_appography_data.to_csv(local_database + 'raw_customer_appography_data.csv', index=False)
df_customer_appography_data = pd.read_csv(local_database + 'raw_customer_appography_data.csv')

# regular-exp 
def extract_and_trim(text):
    return [val.strip() for val in re.findall(r"'([^']*)'", text)]

df_customer_appography_data['app_list_set'] = df_customer_appography_data['app_list_set'].apply(lambda x: extract_and_trim(x))

### Merge

In [11]:
df_active_customer_base.head(1)

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr
0,android,63ff28e9f22ead6bac895265,20,12,8,8,11,1,10


In [12]:
df_customer_segments.head(1)

,customer_id,ltr_segment,life_time_rides,life_time_stage,service_affinity
0,6064a0c0193b5c16f90563d8,UNKNOWN,0,UNKNOWN,UNKNOWN


In [13]:
df_customer_demo_data.head(1)

,customer_id,age,gender
0,64b7c46ab5f93bd98388dec9,0,MALE


In [14]:
df_customer_use_case_data.head(1)

,customer_id,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,63bd2c0a4fd273663b4be401,office,place_of_worship,NaN,NaN


In [15]:
df_customer_appography_data.head(1)

,customer_id,app_list_set
0,5737c6adddbec2203f73316d,"[whatsapp, paytm money, samsung galaxy wearabl..."


In [16]:
start_time = time.time()
print('Merging 1.. ')
merge1 = pd.merge(df_active_customer_base,df_customer_segments,how='left',on = ['customer_id'])

print('Merging 2..')
merge2 = pd.merge(merge1,df_customer_demo_data,how='left',on = ['customer_id'])

print('Merging 3..')
merge3 = pd.merge(merge2,df_customer_use_case_data,how='left',on = ['customer_id'])

print('Merging 4..')
merge4 = pd.merge(merge3,df_customer_appography_data,how='left',on = ['customer_id'])

df_all_customer = merge4

del merge1
del merge2
del merge3
del merge4

end_time = time.time()
execution_time = round(((end_time - start_time)*1.00/60.00), 2)
print("Execution time:", execution_time, "mins")
df_all_customer.head(1)

Merging 1.. 
Merging 2..
Merging 3..
Merging 4..
Execution time: 1.35 mins


,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,android,63ff28e9f22ead6bac895265,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,0,MALE,leisure,NaN,residential,NaN,"[bookmyshow, youtube, jio cinema, google maps,..."


In [17]:
# df_all_customer.to_csv(local_database + 'processed_all_customer.csv', index=False)
# df_all_customer = pd.read_csv(local_database + 'processed_all_customer.csv')

# regular-exp 
# def extract_and_trim(text):
#     return [val.strip() for val in re.findall(r"'([^']*)'", text)]

# df_all_customer['app_list_set'] = df_all_customer['app_list_set'].apply(lambda x: extract_and_trim(x))

In [18]:
df_all_customer.head(1)

,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,age,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_list_set
0,android,63ff28e9f22ead6bac895265,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,0,MALE,leisure,NaN,residential,NaN,"[bookmyshow, youtube, jio cinema, google maps,..."


In [19]:
df_all_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2766893 entries, 0 to 2766892
Data columns (total 20 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   platform                      object 
 1   customer_id                   object 
 2   fe_count                      int64  
 3   rr_count                      int64  
 4   net_count                     int64  
 5   morning_fe                    int64  
 6   evening_fe                    int64  
 7   morning_rr                    int64  
 8   evening_rr                    int64  
 9   ltr_segment                   object 
 10  life_time_rides               float64
 11  life_time_stage               object 
 12  service_affinity              object 
 13  age                           int64  
 14  gender                        object 
 15  first_half_drop_primary       object 
 16  first_half_drop_secondary     object 
 17  second_half_pickup_primary    object 
 18  second_half_pickup_sec

In [20]:
df_all_customer['fe_count'] = df_all_customer['fe_count'].astype('int32')
df_all_customer['rr_count'] = df_all_customer['rr_count'].astype('int32')
df_all_customer['net_count'] = df_all_customer['net_count'].astype('int32')
df_all_customer['morning_fe'] = df_all_customer['morning_fe'].astype('int32')
df_all_customer['evening_fe'] = df_all_customer['evening_fe'].astype('int32')
df_all_customer['morning_rr'] = df_all_customer['morning_rr'].astype('int32')
df_all_customer['evening_rr'] = df_all_customer['evening_rr'].astype('int32')
df_all_customer['age'] = df_all_customer['age'].astype('int32')

df_all_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2766893 entries, 0 to 2766892
Data columns (total 20 columns):
 #   Column                        Dtype  
---  ------                        -----  
 0   platform                      object 
 1   customer_id                   object 
 2   fe_count                      int32  
 3   rr_count                      int32  
 4   net_count                     int32  
 5   morning_fe                    int32  
 6   evening_fe                    int32  
 7   morning_rr                    int32  
 8   evening_rr                    int32  
 9   ltr_segment                   object 
 10  life_time_rides               float64
 11  life_time_stage               object 
 12  service_affinity              object 
 13  age                           int32  
 14  gender                        object 
 15  first_half_drop_primary       object 
 16  first_half_drop_secondary     object 
 17  second_half_pickup_primary    object 
 18  second_half_pickup_sec

## Distribution check
### Platform customer distribution

In [21]:
## platform customer distribution
df_temp = df_all_customer.groupby(['platform']).agg(customers = ('customer_id', 'nunique')).reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,customers,% distribution
0,android,2270829,82.07
1,ios,496064,17.93


### Age customer distribution

In [22]:
df_all_customer['age_presence'] = np.where(df_all_customer['age'] > 0, 'AVAILABLE', 'NA')

df_temp = df_all_customer\
                .groupby(['age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,age_presence,customers,% distribution
0,AVAILABLE,955338,34.61
1,NA,1805166,65.39


In [23]:
df_temp = df_all_customer\
                .groupby(['platform', 'age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,age_presence,customers,total,% distribution
0,android,AVAILABLE,755034,2270829,33.25
1,android,NA,1515795,2270829,66.75
2,ios,AVAILABLE,203139,496064,40.95
3,ios,NA,292925,496064,59.05


### Gender distribution

In [24]:
df_temp = df_all_customer\
                .groupby(['gender'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,gender,customers,% distribution
0,FEMALE,753286,27.29
1,MALE,1926225,69.78
2,OTHERS,8657,0.31
3,UNKNOWN,72336,2.62


In [25]:
df_temp = df_all_customer\
                .groupby(['platform', 'gender'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

,platform,gender,customers,total,% distribution
0,android,FEMALE,592616,2270829,26.10
1,android,MALE,1599463,2270829,70.44
2,android,OTHERS,7745,2270829,0.34
3,android,UNKNOWN,71005,2270829,3.13
4,ios,FEMALE,162571,496064,32.77
5,ios,MALE,331234,496064,66.77
6,ios,OTHERS,927,496064,0.19
7,ios,UNKNOWN,1332,496064,0.27


### Appography distribution

In [26]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_list_set'].isna() == True, 'NA', 'AVAILABLE')

df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'appo_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,appo_presence,customers,% distribution
0,android,AVAILABLE,1877684,82.69
1,android,NA,393145,17.31


### Opportunity Size

In [27]:
df_all_customer['appo_presence'] = np.where(df_all_customer['app_list_set'].isna() == True, 'NA', 'AVAILABLE')

df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'age_presence', 'appo_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

,platform,age_presence,appo_presence,customers,% distribution
0,android,AVAILABLE,AVAILABLE,623111,27.44
1,android,AVAILABLE,NA,131923,5.81
2,android,NA,AVAILABLE,1254573,55.25
3,android,NA,NA,261222,11.50


In [28]:
df_temp.pivot(index ='age_presence' , columns ='appo_presence', values =['% distribution'])

% distribution       
appo_presence      AVAILABLE     NA
age_presence                       
AVAILABLE              27.44   5.81
NA                     55.25  11.50

## One-Hot-Encoder

In [29]:
categorical_columns = ['gender', 'ltr_segment', 'life_time_stage', 'service_affinity', 
                       'first_half_drop_primary', 'first_half_drop_secondary',
                       'second_half_pickup_primary', 'second_half_pickup_secondary']

def one_hot_encoder(df):
    df_categorical = df[categorical_columns].copy()
    
    for column_name in categorical_columns:
        df_categorical[column_name] = df_categorical[column_name].fillna(column_name + '_unknown')
    
    # Initialize the OneHotEncoder
    encoder = OneHotEncoder()
    
    # Fit and transform the categorical columns
    encoded_categorical = encoder.fit_transform(df_categorical)
    
    # Convert the encoded result into a DataFrame
    encoded_df = pd.DataFrame(encoded_categorical.toarray(), 
                              columns=encoder.get_feature_names_out(input_features=categorical_columns))
    
    # Now drop the original categorical columns from the original DataFrame
    df = df.drop(columns=categorical_columns)

    # Concatenate the original DataFrame with the encoded DataFrame
    df = pd.concat([df, encoded_df], axis=1)
    
    
    return df

In [30]:
df_one_hot_encoded = one_hot_encoder(df_all_customer)
print(df_one_hot_encoded.shape)
df_one_hot_encoded['life_time_rides'].fillna(0, inplace=True)
df_one_hot_encoded.head(3)

(2766893, 89)


,platform,customer_id,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,life_time_rides,age,app_list_set,age_presence,appo_presence,gender_FEMALE,gender_MALE,gender_OTHERS,gender_UNKNOWN,ltr_segment_HH,ltr_segment_PHH,ltr_segment_UNKNOWN,ltr_segment_ltr_segment_unknown,life_time_stage_CHURN_OTB,life_time_stage_COMMITTED,life_time_stage_DETOX,...,first_half_drop_secondary_transit_station,second_half_pickup_primary_Unknown,second_half_pickup_primary_educational,second_half_pickup_primary_food,second_half_pickup_primary_govt_amenity,second_half_pickup_primary_health_and_personal,second_half_pickup_primary_household_needs,second_half_pickup_primary_leisure,second_half_pickup_primary_office,second_half_pickup_primary_place_of_worship,second_half_pickup_primary_residential,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_primary_transit_station,second_half_pickup_secondary_Unknown,second_half_pickup_secondary_educational,second_half_pickup_secondary_food,second_half_pickup_secondary_govt_amenity,second_half_pickup_secondary_health_and_personal,second_half_pickup_secondary_household_needs,second_half_pickup_secondary_leisure,second_half_pickup_secondary_office,second_half_pickup_secondary_place_of_worship,second_half_pickup_secondary_residential,second_half_pickup_secondary_second_half_pickup_secondary_unknown,second_half_pickup_secondary_transit_station
0,android,63ff28e9f22ead6bac895265,20,12,8,8,11,1,10,37.0,0,"[bookmyshow, youtube, jio cinema, google maps,...",NA,AVAILABLE,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,android,5cb7f15554bc7263ff3f4e73,61,23,17,34,6,9,4,95.0,28,"[google photos, pinterest, snapchat, youtube, ...",AVAILABLE,AVAILABLE,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,android,62428c959d4b75f09cb3a511,157,103,30,109,40,72,27,248.0,0,"[zee5, meesho, youtube, flipkart, gmail, googl...",NA,AVAILABLE,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,...,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [31]:
df_one_hot_encoded.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr',
       'life_time_rides', 'age', 'app_list_set', 'age_presence',
       'appo_presence', 'gender_FEMALE', 'gender_MALE', 'gender_OTHERS',
       'gender_UNKNOWN', 'ltr_segment_HH', 'ltr_segment_PHH',
       'ltr_segment_UNKNOWN', 'ltr_segment_ltr_segment_unknown',
       'life_time_stage_CHURN_OTB', 'life_time_stage_COMMITTED',
       'life_time_stage_DETOX', 'life_time_stage_DORMANT',
       'life_time_stage_HANDHOLDING', 'life_time_stage_HOOK',
       'life_time_stage_SOFT_CHURN', 'life_time_stage_SUSTENANCE',
       'life_time_stage_UNKNOWN', 'life_time_stage_life_time_stage_unknown',
       'service_affinity_AUTO_CAB', 'service_affinity_LINK_AUTO',
       'service_affinity_LINK_CAB', 'service_affinity_NO_AFFINITY',
       'service_affinity_ONLY_AUTO', 'service_affinity_ONLY_CAB',
       'service_affinity_ONLY_LINK', 'service_affinity_UNKNOWN',
      

In [32]:
encoded_columns = ['customer_id', 
#                    'fe_count', 'rr_count', 'net_count',
#                    'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', # 'life_time_rides', 
#                    'gender_FEMALE', 'gender_MALE', 'gender_OTHERS','gender_UNKNOWN', 
                   'ltr_segment_HH', 'ltr_segment_PHH', 'ltr_segment_UNKNOWN', 
                   'life_time_stage_CHURN_OTB', 'life_time_stage_COMMITTED', 'life_time_stage_DETOX', 
                   'life_time_stage_DORMANT', 'life_time_stage_HANDHOLDING', 'life_time_stage_HOOK',
                   'life_time_stage_SOFT_CHURN', 'life_time_stage_SUSTENANCE', 'life_time_stage_UNKNOWN', 
                   'service_affinity_AUTO_CAB', 'service_affinity_LINK_AUTO','service_affinity_LINK_CAB', 
                   'service_affinity_NO_AFFINITY', 'service_affinity_ONLY_AUTO', 'service_affinity_ONLY_CAB',
                   'service_affinity_ONLY_LINK', 'service_affinity_UNKNOWN',
                   'first_half_drop_primary_Unknown', 
                   'first_half_drop_primary_educational',
                   'first_half_drop_primary_first_half_drop_primary_unknown',
                   'first_half_drop_primary_food', 'first_half_drop_primary_govt_amenity',
                   'first_half_drop_primary_health_and_personal',
                   'first_half_drop_primary_household_needs',
                   'first_half_drop_primary_leisure', 'first_half_drop_primary_office',
                   'first_half_drop_primary_place_of_worship',
                   'first_half_drop_primary_residential',
                   'first_half_drop_primary_transit_station',
                   'first_half_drop_secondary_Unknown',
                   'first_half_drop_secondary_educational',
                   'first_half_drop_secondary_first_half_drop_secondary_unknown',
                   'first_half_drop_secondary_food',
                   'first_half_drop_secondary_govt_amenity',
                   'first_half_drop_secondary_health_and_personal',
                   'first_half_drop_secondary_household_needs',
                   'first_half_drop_secondary_leisure', 'first_half_drop_secondary_office',
                   'first_half_drop_secondary_place_of_worship',
                   'first_half_drop_secondary_residential',
                   'first_half_drop_secondary_transit_station',
                   'second_half_pickup_primary_Unknown',
                   'second_half_pickup_primary_educational',
                   'second_half_pickup_primary_food',
                   'second_half_pickup_primary_govt_amenity',
                   'second_half_pickup_primary_health_and_personal',
                   'second_half_pickup_primary_household_needs',
                   'second_half_pickup_primary_leisure',
                   'second_half_pickup_primary_office',
                   'second_half_pickup_primary_place_of_worship',
                   'second_half_pickup_primary_residential',
                   'second_half_pickup_primary_second_half_pickup_primary_unknown',
                   'second_half_pickup_primary_transit_station',
                   'second_half_pickup_secondary_Unknown',
                   'second_half_pickup_secondary_educational',
                   'second_half_pickup_secondary_food',
                   'second_half_pickup_secondary_govt_amenity',
                   'second_half_pickup_secondary_health_and_personal',
                   'second_half_pickup_secondary_household_needs',
                   'second_half_pickup_secondary_leisure',
                   'second_half_pickup_secondary_office',
                   'second_half_pickup_secondary_place_of_worship',
                   'second_half_pickup_secondary_residential',
                   'second_half_pickup_secondary_second_half_pickup_secondary_unknown',
                   'second_half_pickup_secondary_transit_station']

df_one_hot_encoded = df_one_hot_encoded[encoded_columns]

## Triangulation 1 - Appography Age Bins Classification

In [33]:
df_all_customer.columns

Index(['platform', 'customer_id', 'fe_count', 'rr_count', 'net_count',
       'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 'ltr_segment',
       'life_time_rides', 'life_time_stage', 'service_affinity', 'age',
       'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
       'second_half_pickup_primary', 'second_half_pickup_secondary',
       'app_list_set', 'age_presence', 'appo_presence'],
      dtype='object')

In [34]:
df_triangulation_1 = df_all_customer[(df_all_customer['platform'].isin(['android']) 
                                      & 
                                      df_all_customer['appo_presence'].isin(['AVAILABLE']))]\
                                    [['customer_id', 'age', 'appo_presence','app_list_set',
                                      'fe_count', 'rr_count', 'net_count',
                                      'morning_fe', 'evening_fe', 'morning_rr', 'evening_rr', 
                                      'ltr_segment', 'life_time_rides', 'life_time_stage', 'service_affinity',
                                      'gender', 'first_half_drop_primary', 'first_half_drop_secondary',
                                      'second_half_pickup_primary', 'second_half_pickup_secondary',
                                     ]]

df_triangulation_1.head(5)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,63ff28e9f22ead6bac895265,0,AVAILABLE,"[bookmyshow, youtube, jio cinema, google maps,...",20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN
1,5cb7f15554bc7263ff3f4e73,28,AVAILABLE,"[google photos, pinterest, snapchat, youtube, ...",61,23,17,34,6,9,4,PHH,95.0,COMMITTED,ONLY_AUTO,MALE,health_and_personal,place_of_worship,office,residential
2,62428c959d4b75f09cb3a511,0,AVAILABLE,"[zee5, meesho, youtube, flipkart, gmail, googl...",157,103,30,109,40,72,27,PHH,248.0,COMMITTED,ONLY_AUTO,FEMALE,household_needs,transit_station,household_needs,transit_station
4,5fbaaf067e483f4a13fd9b9e,28,AVAILABLE,"[whatsapp, swiggy, google maps, youtube, spoti...",85,45,15,13,64,9,32,PHH,87.0,COMMITTED,ONLY_AUTO,MALE,transit_station,office,residential,health_and_personal
5,64d0d9ebc810cc925f36ceea,0,AVAILABLE,"[truecaller, google maps, youtube, flipkart, m...",111,72,50,27,84,19,53,PHH,119.0,COMMITTED,ONLY_AUTO,FEMALE,residential,transit_station,residential,educational


### Explode

In [35]:
## Explode list into individual rows

df_appography_explode = df_triangulation_1.explode('app_list_set')
df_appography_explode['app_list_set'] = df_appography_explode['app_list_set'].str.lower()

df_appography_explode.head(4)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN
0,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN
0,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN
0,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN


### Mobile app to category mapping

In [36]:
category_list = pd.read_csv('/Users/rapido/local-datasets/customer/appography/appography_v1/category_list_v1.csv')
category_list['app_name'] = category_list['app_name'].str.lower()
category_list['categories_l1'] = category_list['categories_l1'].str.lower()
category_list['categories_l2'] = category_list['categories_l2'].str.lower()
category_list = category_list[['app_name', 'categories_l1', 'categories_l2']]
print(category_list.shape)
print(category_list.categories_l1.unique())
print(category_list.categories_l2.unique())
category_list.head(1)

(433, 3)
['educational' 'ott' 'delivery_grocery' 'tools' 'office' 'dating' 'news'
 'devotional' 'travel_metro' 'gaming' 'travel_bookings'
 'finance_transactions' 'ecommerce' 'streaming_music' 'finance_investment'
 'health' 'fantasy_sports' 'delivery_food' 'home_security' 'finance_news'
 'travel_ridehailing' 'social' 'telecom' 'entertainment' 'news_finance'
 'wearable' 'driver_app' 'service']
['educational' 'rest' 'office' 'ride haling' 'gaming' 'finance_investment'
 'driver_app']


,app_name,categories_l1,categories_l2
0,ignou e-content,educational,educational


In [37]:
## Derived category

core_education = ['aakash','chegg study','google classroom','microsoft math solver','simplilearn','moodle','vedantu','vidyakul','diksha','e-pg pathshala','mycbseguide','physics wallah']
core_office = ['slack','asana','confluence','jira','cisco','github','google analytics','miro','trello','zoho mail','zoho meeting']
top_categories = ['dating', 'delivery_food', 'driver_app', 'entertainment', 'finance_investment', 'gaming', 'office', 'social', 'streaming_music', 'travel_bookings', 'travel_ridehailing']


category_list['top_categories'] = np.where(category_list['app_name'].isin(core_education), 'core_education',  #category_list['app_name'],
                                    np.where(category_list['app_name'].isin(core_office), 'core_office', #category_list['app_name'],
                                        np.where(category_list['categories_l1'].isin(top_categories), category_list['categories_l1'], 'rest'
                                            )))

category_list['last_categories'] = np.where(category_list['app_name'].isin(core_education), 'core_education',  #category_list['app_name'],
                                    np.where(category_list['app_name'].isin(core_office), 'core_office', #category_list['app_name'],
                                        np.where(category_list['categories_l1'].isin(top_categories), category_list['categories_l1'], category_list['categories_l1']
                                            )))

In [38]:
top_apps_list = category_list[category_list['top_categories'] != 'rest']['app_name'].values.tolist()

In [39]:
## Top apps
# df_main_apps_explode = df_appography_explode[df_appography_explode['app_list_set'].isin(top_apps_list)]

df_main_apps_explode = df_appography_explode

### Merge

In [40]:
df_main_apps_explode.head(2)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN
0,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN


In [41]:
category_list.head(2)

,app_name,categories_l1,categories_l2,top_categories,last_categories
0,ignou e-content,educational,educational,rest,educational
1,aha,ott,rest,rest,ott


In [42]:
df_main_app_category =  pd.merge(df_main_apps_explode, 
                                 category_list, 
                                 how='left', 
                                 left_on='app_list_set', 
                                 right_on='app_name'
                              )
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,categories_l1,categories_l2,top_categories,last_categories
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,bookmyshow,entertainment,rest,entertainment,entertainment
1,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,youtube,tools,rest,rest,tools
2,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,jio cinema,ott,rest,rest,ott
3,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,google maps,tools,rest,rest,tools


#### Distribution check on categories

In [43]:
total_customers = df_main_app_category.customer_id.nunique()
total_customers

df_temp = df_main_app_category\
            .groupby(['categories_l1'])\
            .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                 customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,categories_l1,app_list,customers,%
0,tools,"[truecaller, gmail, adobe scan, youtube, picsa...",1877625,100.00
1,social,"[facebook, whatsapp, skype, line, roposo, hike...",1875178,99.87
2,ott,"[netflix, zee5, alt balaji, jio cinema, sonyli...",1527173,81.33
3,ecommerce,"[ajio, myntra, pharmeasy, paytm mall, amazon s...",1501596,79.97
4,travel_ridehailing,"[driveu driver app, namma yatri, quick ride, u...",1335369,71.12
5,finance_transactions,"[axis mobile, freecharge, mobikwik, cred, payt...",1264156,67.33
6,streaming_music,"[wynk music, soundcloud, amazon music, gaana, ...",1254092,66.79
7,delivery_food,"[swiggy, uber eats, zomato, dunzo]",1153487,61.43
8,office,"[zoom, cisco, onedrive, asana, jira, slack, go...",1067225,56.84
9,finance_investment,"[hdfc mobile banking, nps, angel broking, grow...",662382,35.28


#### Most Common used apps

In [44]:
df_temp = df_main_app_category\
            .groupby(['app_list_set'])\
            .agg(customers = ('customer_id','nunique'))\
            .sort_values(['customers'],ascending=False)\
            .reset_index()
df_temp['%'] =  (df_temp['customers']*100.00/total_customers).round(2)
df_temp

,app_list_set,customers,%
0,google maps,1877029,99.97
1,gmail,1873112,99.76
2,youtube,1873057,99.75
3,whatsapp,1830156,97.47
4,google photos,1775553,94.56
5,instagram,1523428,81.13
6,facebook,1310924,69.82
7,flipkart,1273245,67.81
8,truecaller,1221064,65.03
9,google calendar,1187551,63.25


In [45]:
most_common_apps = ['google maps', 'gmail', 'youtube', 'whatsapp', 'clock', 'calculator',
                    'google calendar', 'google photos']

### Way 1 (All Apps Level)

In [46]:
## Unwanted features

#### Transform Categories 

In [47]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,categories_l1,categories_l2,top_categories,last_categories
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,bookmyshow,entertainment,rest,entertainment,entertainment
1,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,youtube,tools,rest,rest,tools
2,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,jio cinema,ott,rest,rest,ott
3,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,google maps,tools,rest,rest,tools


In [48]:
df_customer_mapped_v1 = df_main_app_category[~df_main_app_category['app_list_set'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))),
                             top_categories = ('top_categories', lambda x: list(set(x))))\
                        .reset_index()

df_customer_mapped_v1.head(4)

,customer_id,age,app_list,categories_l1,top_categories
0,5737c6adddbec2203f73316d,0,"[facebook, truecaller, cred, onedrive, picsart...","[social, streaming_music, tools, finance_trans...","[social, streaming_music, driver_app, office, ..."
1,5737c6aeddbec2203f733176,34,"[netflix, facebook, microsoft 365, uber, messe...","[social, delivery_grocery, tools, finance_tran...","[social, office, finance_investment, delivery_..."
2,5737c6b0ddbec2203f733182,28,"[facebook, truecaller, zomato, dunzo, wynk mus...","[social, entertainment, streaming_music, tools...","[social, entertainment, streaming_music, drive..."
3,5737c6b1ddbec2203f73318b,33,"[facebook, truecaller, zoom, zomato, google ne...","[social, travel_bookings, news, delivery_groce...","[social, travel_bookings, driver_app, office, ..."


In [49]:
unique_categories = set(category for sublist in df_customer_mapped_v1['app_list'] for category in sublist)
unique_categories_list = list(unique_categories)
print("Count of unique apps:", len(unique_categories))
sorted(unique_categories)

Count of unique apps: 212


['aakash',
 'aastha tv',
 'accuweather',
 'adobe lightroom',
 'adobe scan',
 'aha',
 'airbnb',
 'aisle',
 'ajio',
 'alt balaji',
 'amazon music',
 'amazon prime video',
 'amazon shopping',
 'angel broking',
 'asana',
 'axis mobile',
 'bansal classes',
 'bigbasket',
 'blusmart',
 'booking.com',
 'bookmyshow',
 'borzo driver app',
 'brainly',
 'bumble',
 'byju',
 'caclubindia',
 'chegg study',
 'cisco',
 'cleartrip',
 'club factory',
 'coffee meets bagel',
 'colegeduniya',
 'collegedekho',
 'confluence',
 'course hero',
 'coursera',
 'cred',
 'cricplay',
 'diksha',
 'disney+ hotstar',
 'doubtnut',
 'driveu driver app',
 'dropbox',
 'dunzo',
 'dunzo driver app',
 'duolingo',
 'e-pg pathshala',
 'economic times',
 'edx',
 'embibe',
 'facebook',
 'feedly',
 'fiitjee',
 'file commander',
 'firstcry',
 'fitbit',
 'fiton',
 'flipboard',
 'flipkart',
 'fossil smartwatches',
 'freecharge',
 'gaana',
 'garmin connect',
 'geeks for geeks',
 'github',
 'goibibo',
 'goodreads',
 'google analytics',


In [50]:
for category in unique_categories:
    df_customer_mapped_v1[category] = df_customer_mapped_v1['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v1.head(1)

,customer_id,age,app_list,categories_l1,top_categories,garmin connect,nike training club,tinder,aakash,cred,aha,groww,wynk music,netflix,uber driver app,myntra,skype,feedly,hike,aisle,khan academy,aastha tv,my study life,my11circle,microsoft teams,...,indigo,shiksha mitra,borzo driver app,redbus,google slides,chegg study,course hero,jiomart,zoho mail,edx,driveu driver app,amazon music,mlb,mobikwik,healthifyme,photomath,paytm money,nhl,snapchat,dunzo driver app,quickride,meesho,airbnb,ms authenticator,brainly
0,5737c6adddbec2203f73316d,0,"[facebook, truecaller, cred, onedrive, picsart...","[social, streaming_music, tools, finance_trans...","[social, streaming_music, driver_app, office, ...",0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0


In [51]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

# bins = [15, 20, 25, 35, 61]
# labels = ['15-20', '21-25',  '26-35', '36-60']

df_customer_mapped_v1['age_group'] = pd.cut(df_customer_mapped_v1['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [52]:
main_list = ['customer_id', 'age', 'age_group' ,'categories_l1']
unique_column_list = main_list + unique_categories_list

In [53]:
df_customer_mapped_v1 = df_customer_mapped_v1[unique_column_list]
df_customer_mapped_v1.head(1)

,customer_id,age,age_group,categories_l1,garmin connect,nike training club,tinder,aakash,cred,aha,groww,wynk music,netflix,uber driver app,myntra,skype,feedly,hike,aisle,khan academy,aastha tv,my study life,my11circle,microsoft teams,voonik,...,indigo,shiksha mitra,borzo driver app,redbus,google slides,chegg study,course hero,jiomart,zoho mail,edx,driveu driver app,amazon music,mlb,mobikwik,healthifyme,photomath,paytm money,nhl,snapchat,dunzo driver app,quickride,meesho,airbnb,ms authenticator,brainly
0,5737c6adddbec2203f73316d,0,NaN,"[social, streaming_music, tools, finance_trans...",0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,1,0,0,0,0,0,0


#### OneHotEncoder

In [54]:
# Adding required additional info by one-hot-encoding 

df_customer_mapped_v1 = pd.merge(df_customer_mapped_v1, 
                                 df_one_hot_encoded,
                                 how='left',
                                 left_on='customer_id',
                                 right_on='customer_id'
                                )
# df_customer_mapped_v1['life_time_rides'].fillna(0, inplace=True)

df_customer_mapped_v1.head(1)

,customer_id,age,age_group,categories_l1,garmin connect,nike training club,tinder,aakash,cred,aha,groww,wynk music,netflix,uber driver app,myntra,skype,feedly,hike,aisle,khan academy,aastha tv,my study life,my11circle,microsoft teams,voonik,...,first_half_drop_secondary_transit_station,second_half_pickup_primary_Unknown,second_half_pickup_primary_educational,second_half_pickup_primary_food,second_half_pickup_primary_govt_amenity,second_half_pickup_primary_health_and_personal,second_half_pickup_primary_household_needs,second_half_pickup_primary_leisure,second_half_pickup_primary_office,second_half_pickup_primary_place_of_worship,second_half_pickup_primary_residential,second_half_pickup_primary_second_half_pickup_primary_unknown,second_half_pickup_primary_transit_station,second_half_pickup_secondary_Unknown,second_half_pickup_secondary_educational,second_half_pickup_secondary_food,second_half_pickup_secondary_govt_amenity,second_half_pickup_secondary_health_and_personal,second_half_pickup_secondary_household_needs,second_half_pickup_secondary_leisure,second_half_pickup_secondary_office,second_half_pickup_secondary_place_of_worship,second_half_pickup_secondary_residential,second_half_pickup_secondary_second_half_pickup_secondary_unknown,second_half_pickup_secondary_transit_station
0,5737c6adddbec2203f73316d,0,NaN,"[social, streaming_music, tools, finance_trans...",0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [55]:
# Find columns with NaN values
columns_with_nan = df_customer_mapped_v1.columns[df_customer_mapped_v1.isna().any()].tolist()

# Filter DataFrame to get only columns with NaN values
df_with_nan = df_customer_mapped_v1[columns_with_nan]
df_with_nan.isnull().sum()

age_group    1255461
dtype: int64

In [56]:
# df_customer_mapped_v1['life_time_rides'].fillna(0, inplace=True)

#### Stratified Sampling & Model

In [57]:
df_customer_mapped_v1_age_w = df_customer_mapped_v1[~df_customer_mapped_v1['age_group'].isna()]
df_customer_mapped_v1_age_na = df_customer_mapped_v1[df_customer_mapped_v1['age_group'].isna()]
df_customer_mapped_v1_age_w.drop(columns = ['customer_id', 'categories_l1', 'age'], axis=1, inplace=True)
df_customer_mapped_v1_age_na.drop(columns = ['customer_id', 'categories_l1', 'age'], axis=1, inplace=True)

target_classes = df_customer_mapped_v1_age_w['age_group'].unique().tolist()

def evaluate_target_vs_rest(df, target_column):
    target_classes = df[target_column].unique().tolist()
    target_classes.sort()
    
    for target_class in target_classes:
        # Create a binary target column where target_class is 1 and rest are 0
        df['target'] = df[target_column].apply(lambda x: 1 if x == target_class else 0)

        # Stratified sampling to maintain class distribution
        X_train, X_test, y_train, y_test = train_test_split(df.drop(columns=[target_column, 'target']), 
                                                            df['target'], 
                                                            test_size=0.2, 
                                                            stratify=df['target'], 
                                                            random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        # Make predictions
        y_pred = rf_model.predict(X_test)

        # Evaluate model performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Output results
        print(f"Results for class {target_class} vs Rest:")
        print("Accuracy : ",  round(accuracy * 100.00, 2), "%")
        print("Precision : ", round(precision * 100.00, 2), "%")
        print("Recall : ", round(recall * 100.00, 2), "%")
        print("F1 Score : ", round(f1 * 100.00, 2), "%")
        print("---------------------------------------")
        
        # Drop the target column for the next iteration
        df.drop(columns=['target'], inplace=True)

In [58]:
def evaluate_target_vs_rest(df, target_column):
    target_classes = df[target_column].unique().tolist()
    target_classes.sort()
    
    for target_class in target_classes:
        
        # backup df  
        df_temp = df
        
        # Split the bins target_class vs Rest 
        df_target = df_temp[df_temp[target_column] == target_class]
        df_rest = df_temp[df_temp[target_column] != target_class]
        
        df_other_sample = df_rest.sample(n=df_target.shape[0], random_state=1)
        
        df_combined = pd.concat([df_target, df_other_sample], ignore_index=True)
        # print(df_target.shape)
        # print(df_rest.shape)
        # print(df_other_sample.shape)
        # print(df_combined.shape)
        
        # Create a binary target column where target_class is 1 and rest are 0
        df_combined['target'] = df_combined[target_column].apply(lambda x: 1 if x == target_class else 0)

        # train_test_split
        X_train, X_test, y_train, y_test = train_test_split(df_combined.drop(columns=[target_column, 'target']), 
                                                            df_combined['target'], 
                                                            test_size=0.2, 
                                                            random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        # Extract feature importances
        feature_importances = rf_model.feature_importances_

        # Get the indices of top features
        top_feature_indices = np.argsort(feature_importances)[::-1][:20]  # Adjust 5 as needed

        # Get the names of top features
        top_features = X_train.columns[top_feature_indices]

        # Make predictions
        y_pred = rf_model.predict(X_test)

        # Evaluate model performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Output results
        print(f"Results for class {target_class} vs Rest:")
        print("Accuracy : ",  round(accuracy * 100.00, 2), "%")
        print("Precision : ", round(precision * 100.00, 2), "%")
        print("Recall : ", round(recall * 100.00, 2), "%")
        print("F1 Score : ", round(f1 * 100.00, 2), "%")
        print("Top Features:")
        for feature in top_features:
            print(feature)
        print("---------------------------------------")
        
        # Drop the target column for the next iteration
        df_combined.drop(columns=['target'], inplace=True)

In [59]:
evaluate_target_vs_rest(df_customer_mapped_v1_age_w, 'age_group')

Results for class 15-20 vs Rest:
Accuracy :  69.94 %
Precision :  70.13 %
Recall :  68.04 %
F1 Score :  69.07 %
Top Features:
cred
snapchat
hdfc mobile banking
paytm money
linkedin
messenger
spotify
axis mobile
telegram
jio cinema
disney+ hotstar
ola
intune  company portal
uber
flipkart
truecaller
google news
netflix
wynk music
meesho
---------------------------------------
Results for class 21-25 vs Rest:
Accuracy :  70.49 %
Precision :  70.32 %
Recall :  70.55 %
F1 Score :  70.44 %
Top Features:
snapchat
messenger
google classroom
telegram
cred
spotify
instagram
jio cinema
google news
service_affinity_ONLY_LINK
wynk music
facebook
uber
truecaller
paytm money
netflix
disney+ hotstar
first_half_drop_primary_first_half_drop_primary_unknown
linkedin
flipkart
---------------------------------------
Results for class 26-30 vs Rest:
Accuracy :  60.72 %
Precision :  60.92 %
Recall :  60.31 %
F1 Score :  60.61 %
Top Features:
truecaller
google news
snapchat
spotify
paytm money
telegram
wynk m

In [66]:
features_to_test = ['age_group',
                    'snapchat','cred','hdfc mobile banking','paytm money','linkedin','axis mobile',
                    'messenger','spotify','telegram','jio cinema','ola','uber','disney+ hotstar',
                    'amazon prime video', 'sharechat', 'bookmyshow', 'meesho', ''
#                     'google news',
                    'flipkart','truecaller','netflix',
                    'intune  company portal','wynk music', 'zomato', 'groww','facebook',
                    'zoom', 'mx takatak', 'myntra', 'instagram','picsart'
                   ] 

df1_analysis = df_customer_mapped_v1_age_w[features_to_test]
df1_analysis

,age_group,snapchat,cred,hdfc mobile banking,paytm money,linkedin,axis mobile,messenger,spotify,telegram,jio cinema,ola,uber,disney+ hotstar,amazon prime video,sharechat,bookmyshow,meesho,flipkart,truecaller,netflix,intune company portal,wynk music,zomato,groww,facebook,zoom,mx takatak,myntra,instagram,picsart
1,31-35,0,1,0,1,0,0,1,0,0,0,1,1,0,0,0,0,0,0,0,1,0,0,1,1,1,0,0,0,1,0
2,26-30,0,0,0,1,0,0,1,0,1,1,1,1,1,1,0,1,0,1,1,0,0,1,1,0,1,0,1,0,1,0
3,31-35,1,0,0,1,0,0,1,0,1,1,1,1,1,1,0,0,0,1,1,0,0,0,1,0,1,1,0,0,1,0
6,26-30,0,1,1,1,0,0,1,0,1,1,0,0,0,0,0,0,1,1,1,0,0,1,0,0,1,0,0,0,1,0
7,36-45,0,0,0,1,0,0,1,0,0,1,0,0,1,0,0,0,0,1,1,0,0,0,1,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1879879,21-25,1,0,0,1,0,1,1,0,0,0,0,0,1,0,0,0,1,1,1,0,0,0,1,0,1,0,0,0,1,0
1879888,26-30,1,0,0,1,0,0,0,0,1,1,0,0,1,0,0,1,0,1,1,1,0,0,0,0,1,0,1,1,1,1
1879949,15-20,1,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,1,0,0,0,0,0,1,0,0,0,0,0
1880042,31-35,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,0,0,0,1,0,0,0,1,1,0,0,0,0,1,0


In [67]:
customer_count = pd.DataFrame(df1_analysis.groupby(df1_analysis.iloc[:, 0]).size(), columns=['customers']).reset_index()
customer_count

,age_group,customers
0,15-20,27615
1,21-25,147273
2,26-30,206446
3,31-35,127391
4,36-45,92601
5,46-60,23608


In [68]:
sum_last_20 = df1_analysis.groupby(df1_analysis.iloc[:, 0])[df1_analysis.columns[-20:]].sum().reset_index()

sum_last_20

,age_group,ola,uber,disney+ hotstar,amazon prime video,sharechat,bookmyshow,meesho,flipkart,truecaller,netflix,intune company portal,wynk music,zomato,groww,facebook,zoom,mx takatak,myntra,instagram,picsart
0,15-20,14801,13178,13197,9263,2550,8588,8485,18521,17853,12257,1215,7161,12791,3460,18358,5873,4699,8574,23443,3974
1,21-25,82013,76850,85040,63414,9531,58108,40964,108215,100259,75511,16219,42256,86051,29602,94530,36891,33916,62319,132002,25047
2,26-30,127757,112646,130832,104683,17506,86499,55079,153597,148613,116182,35646,63718,130091,42861,142285,53796,49878,94647,179625,25249
3,31-35,83186,71371,80470,66084,15730,52659,35153,93298,93487,69832,21985,41636,76604,22081,95993,35903,25230,50927,103097,8971
4,36-45,60695,51579,52992,44870,14673,31610,25797,62939,66720,47156,13241,25805,48115,10702,73081,29507,12640,27837,65457,3755
5,46-60,15885,14100,11722,10045,3122,5909,5821,13724,16569,11198,1720,4565,10441,1864,18382,8461,2177,5533,14695,908


In [69]:
sum_last_20.to_clipboard(index=False)

In [64]:
def evaluate_target_vs_rest_with_pca(df, target_column, n_components=10):
    target_classes = df[target_column].unique().tolist()
    target_classes.sort()
    
    for target_class in target_classes:
        
        # backup df  
        df_temp = df
        
        # Split the bins target_class vs Rest 
        df_target = df_temp[df_temp[target_column] == target_class]
        df_rest = df_temp[df_temp[target_column] != target_class]
        
        df_other_sample = df_rest.sample(n=df_target.shape[0], random_state=1)
        
        df_combined = pd.concat([df_target, df_other_sample], ignore_index=True)
        
        # Create a binary target column where target_class is 1 and rest are 0
        df_combined['target'] = df_combined[target_column].apply(lambda x: 1 if x == target_class else 0)
        
        # Apply PCA
        pca = PCA(n_components=n_components)
        X_pca = pca.fit_transform(df_combined.drop(columns=[target_column, 'target']))

        # train_test_split
        X_train, X_test, y_train, y_test = train_test_split(X_pca, 
                                                            df_combined['target'], 
                                                            test_size=0.2, 
                                                            random_state=42)

        # Train Random Forest model
        rf_model = RandomForestClassifier(random_state=42)
        rf_model.fit(X_train, y_train)

        # Make predictions
        y_pred = rf_model.predict(X_test)

        # Evaluate model performance
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)

        # Output results
        print(f"Results for class {target_class} vs Rest:")
        print("Accuracy : ",  round(accuracy * 100.00, 2), "%")
        print("Precision : ", round(precision * 100.00, 2), "%")
        print("Recall : ", round(recall * 100.00, 2), "%")
        print("F1 Score : ", round(f1 * 100.00, 2), "%")
        print("---------------------------------------")
        
        # Drop the target column for the next iteration
        df_combined.drop(columns=['target'], inplace=True) 

In [65]:
evaluate_target_vs_rest_with_pca(df_customer_mapped_v1_age_w, 'age_group', n_components=10)

Results for class 15-20 vs Rest:
Accuracy :  66.35 %
Precision :  66.45 %
Recall :  64.17 %
F1 Score :  65.29 %
---------------------------------------
Results for class 21-25 vs Rest:
Accuracy :  67.17 %
Precision :  66.81 %
Recall :  67.78 %
F1 Score :  67.29 %
---------------------------------------
Results for class 26-30 vs Rest:
Accuracy :  56.79 %
Precision :  57.22 %
Recall :  54.62 %
F1 Score :  55.89 %
---------------------------------------
Results for class 31-35 vs Rest:
Accuracy :  59.9 %
Precision :  58.97 %
Recall :  63.94 %
F1 Score :  61.36 %
---------------------------------------
Results for class 36-45 vs Rest:
Accuracy :  65.3 %
Precision :  64.08 %
Recall :  69.51 %
F1 Score :  66.69 %
---------------------------------------
Results for class 46-60 vs Rest:
Accuracy :  70.58 %
Precision :  71.1 %
Recall :  69.81 %
F1 Score :  70.45 %
---------------------------------------


#### Feature selection validation

## Other Options
### Way 2 (Category Level)

#### Transform Categories 

In [65]:
df_main_app_category.head(4)

,customer_id,age,appo_presence,app_list_set,fe_count,rr_count,net_count,morning_fe,evening_fe,morning_rr,evening_rr,ltr_segment,life_time_rides,life_time_stage,service_affinity,gender,first_half_drop_primary,first_half_drop_secondary,second_half_pickup_primary,second_half_pickup_secondary,app_name,categories_l1,categories_l2,top_categories,last_categories
0,63ff28e9f22ead6bac895265,0,AVAILABLE,bookmyshow,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,bookmyshow,entertainment,rest,entertainment,entertainment
1,63ff28e9f22ead6bac895265,0,AVAILABLE,youtube,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,youtube,tools,rest,rest,tools
2,63ff28e9f22ead6bac895265,0,AVAILABLE,jio cinema,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,jio cinema,ott,rest,rest,ott
3,63ff28e9f22ead6bac895265,0,AVAILABLE,google maps,20,12,8,8,11,1,10,PHH,37.0,HOOK,ONLY_LINK,MALE,leisure,NaN,residential,NaN,google maps,tools,rest,rest,tools


In [67]:
df_customer_mapped_v2 = df_main_app_category[~df_main_app_category['app_list_set'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(categories_l1 = ('categories_l1', lambda x: list(set(x))))\
                        .reset_index()

df_customer_mapped_v2.head(4)

In [ ]:
v1_unique_categories = set(category for sublist in df_customer_mapped_v2['categories_l1'] for category in sublist)
v1_unique_categories_list = list(v1_unique_categories)
print("Count of unique apps:", len(v1_unique_categories))
sorted(v1_unique_categories)

In [ ]:
for category in v1_unique_categories:
    df_customer_mapped_v2[category] = df_customer_mapped_v2['categories_l1'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v2.head(1)

In [ ]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

# bins = [15, 20, 25, 35, 61]
# labels = ['15-20', '21-25',  '26-35', '36-60']

df_customer_mapped_v2['age_group'] = pd.cut(df_customer_mapped_v2['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [ ]:
v1_main_list = ['customer_id', 'age', 'age_group' ,'categories_l1']
v1_unique_column_list = v1_main_list + v1_unique_categories_list

In [ ]:
df_customer_mapped_v2 = df_customer_mapped_v2[v1_unique_column_list]
df_customer_mapped_v2.head(1)

#### OneHotEncoder

In [ ]:
df_customer_mapped_v2 = pd.merge(df_customer_mapped_v2, 
                                 df_one_hot_encoded,
                                 how='left',
                                 left_on='customer_id',
                                 right_on='customer_id'
                                )
df_customer_mapped_v2.head(1)

In [ ]:
# Find columns with NaN values
columns_with_nan = df_customer_mapped_v2.columns[df_customer_mapped_v2.isna().any()].tolist()

# Filter DataFrame to get only columns with NaN values
df_with_nan = df_customer_mapped_v2[columns_with_nan]
df_with_nan.isnull().sum()

In [ ]:
df_customer_mapped_v2_age_w = df_customer_mapped_v2[~df_customer_mapped_v2['age_group'].isna()]
df_customer_mapped_v2_age_na = df_customer_mapped_v2[df_customer_mapped_v2['age_group'].isna()]
df_customer_mapped_v2_age_w.drop(columns=['customer_id', 'age', 'categories_l1'], axis=1, inplace=True)
df_customer_mapped_v2_age_na.drop(columns=['customer_id', 'age', 'categories_l1'], axis=1, inplace=True)

#### Stratified Sampling & Model

In [ ]:
df_customer_mapped_v2_age_w.head(1)

In [ ]:
evaluate_target_vs_rest(df_customer_mapped_v2_age_w, 'age_group')

In [ ]:
evaluate_target_vs_rest_with_pca(df_customer_mapped_v2_age_w, 'age_group', n_components=10)

#### Feature selection 

### Way 3 (App + Category Level)

#### Transform Categories 

In [ ]:
df_main_app_category.head(4)

In [ ]:
df_customer_mapped_v3 = df_main_app_category[~df_main_app_category['app_list_set'].isin(most_common_apps)]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))))\
                        .reset_index()

df_customer_mapped_v3.head(4)

In [ ]:
v2_unique_categories = set(category for sublist in df_customer_mapped_v3['app_list'] for category in sublist)
v2_unique_categories_list = list(v2_unique_categories)
print("Count of unique apps:", len(v2_unique_categories))
sorted(v2_unique_categories)

In [ ]:
for category in v2_unique_categories:
    df_customer_mapped_v3[category] = df_customer_mapped_v3['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v3.head(1)

In [ ]:
v2a_unique_categories = set(category for sublist in df_customer_mapped_v3['categories_l1'] for category in sublist)
v2a_unique_categories_list = list(v2a_unique_categories)
print("Count of unique apps:", len(v2a_unique_categories))
sorted(v2a_unique_categories)

In [ ]:
for category in v2a_unique_categories:
    df_customer_mapped_v3[category] = df_customer_mapped_v3['categories_l1'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v3.head(1)

In [ ]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

# bins = [15, 20, 25, 35, 61]
# labels = ['15-20', '21-25',  '26-35', '36-60']

df_customer_mapped_v3['age_group'] = pd.cut(df_customer_mapped_v3['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [ ]:
v2_main_list = ['customer_id', 'age', 'age_group' , 'app_list', 'categories_l1']
v2_unique_column_list = v2_main_list + v2_unique_categories_list + v2a_unique_categories_list

In [ ]:
df_customer_mapped_v3 = df_customer_mapped_v3[v2_unique_column_list]
df_customer_mapped_v3.head(1)

#### OneHotEncoder

In [ ]:
df_customer_mapped_v3 = pd.merge(df_customer_mapped_v3, 
                                 df_one_hot_encoded,
                                 how='left',
                                 left_on='customer_id',
                                 right_on='customer_id'
                                )
df_customer_mapped_v3.head(1)

In [ ]:
# Find columns with NaN values
columns_with_nan = df_customer_mapped_v3.columns[df_customer_mapped_v3.isna().any()].tolist()

# Filter DataFrame to get only columns with NaN values
df_with_nan = df_customer_mapped_v3[columns_with_nan]
df_with_nan.isnull().sum()

In [ ]:
df_customer_mapped_v3_age_w = df_customer_mapped_v3[~df_customer_mapped_v3['age_group'].isna()]
df_customer_mapped_v3_age_na = df_customer_mapped_v3[df_customer_mapped_v3['age_group'].isna()]
df_customer_mapped_v3_age_w.drop(columns=['customer_id', 'age', 'categories_l1', 'app_list'], axis=1, inplace=True)
df_customer_mapped_v3_age_na.drop(columns=['customer_id', 'age', 'categories_l1', 'app_list'], axis=1, inplace=True)

#### Model

In [ ]:
df_customer_mapped_v3_age_w.head(1)

In [ ]:
evaluate_target_vs_rest(df_customer_mapped_v3_age_w, 'age_group')

In [ ]:
evaluate_target_vs_rest_with_pca(df_customer_mapped_v3_age_w, 'age_group', n_components=10)

#### Feature selection 

### Way 4 (Top App)

#### Transform Categories 

In [ ]:
df_main_app_category.head(4)

In [ ]:
df_customer_mapped_v4 = df_main_app_category[~df_main_app_category['top_categories'].isin(['rest'])]\
                        .groupby(['customer_id', 'age'])\
                        .agg(app_list = ('app_list_set', lambda x: list(set(x))),
                             categories_l1 = ('categories_l1', lambda x: list(set(x))))\
                        .reset_index()

df_customer_mapped_v4.head(4)

In [ ]:
v3_unique_categories = set(category for sublist in df_customer_mapped_v4['app_list'] for category in sublist)
v3_unique_categories_list = list(v3_unique_categories)
print("Count of unique apps:", len(v3_unique_categories))
sorted(v3_unique_categories)

In [ ]:
for category in v3_unique_categories:
    df_customer_mapped_v4[category] = df_customer_mapped_v4['app_list'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v4.head(1)

In [ ]:
v3a_unique_categories = set(category for sublist in df_customer_mapped_v4['categories_l1'] for category in sublist)
v3a_unique_categories_list = list(v3a_unique_categories)
print("Count of unique apps:", len(v3a_unique_categories))
sorted(v3a_unique_categories)

In [ ]:
for category in v3a_unique_categories:
    df_customer_mapped_v4[category] = df_customer_mapped_v4['categories_l1'].apply(lambda x: 1 if category in x else 0) 
    
df_customer_mapped_v4.head(1)

In [ ]:
bins = [15, 20, 25, 30, 35, 45, 61]
labels = ['15-20', '21-25',  '26-30', '31-35', '36-45', '46-60']

# bins = [15, 20, 25, 35, 61]
# labels = ['15-20', '21-25',  '26-35', '36-60']

df_customer_mapped_v4['age_group'] = pd.cut(df_customer_mapped_v4['age'], 
                                            bins=bins, 
                                            labels=labels, 
                                            right=False)

In [ ]:
v3_main_list = ['customer_id', 'age', 'age_group' , 'app_list', 'categories_l1']
v3_unique_column_list = v3_main_list + v3_unique_categories_list + v3a_unique_categories_list

In [ ]:
df_customer_mapped_v4 = df_customer_mapped_v4[v3_unique_column_list]
df_customer_mapped_v4.head(1)

#### OneHotEncoder

In [ ]:
df_customer_mapped_v4 = pd.merge(df_customer_mapped_v4, 
                                 df_one_hot_encoded,
                                 how='left',
                                 left_on='customer_id',
                                 right_on='customer_id'
                                )
df_customer_mapped_v4.head(1)

In [ ]:
# Find columns with NaN values
columns_with_nan = df_customer_mapped_v4.columns[df_customer_mapped_v4.isna().any()].tolist()

# Filter DataFrame to get only columns with NaN values
df_with_nan = df_customer_mapped_v4[columns_with_nan]
df_with_nan.isnull().sum()

In [ ]:
df_customer_mapped_v4_age_w = df_customer_mapped_v4[~df_customer_mapped_v4['age_group'].isna()]
df_customer_mapped_v4_age_na = df_customer_mapped_v4[df_customer_mapped_v4['age_group'].isna()]
df_customer_mapped_v4_age_w.drop(columns=['customer_id', 'age','categories_l1', 'app_list'], axis=1, inplace=True)
df_customer_mapped_v4_age_na.drop(columns=['customer_id', 'age','categories_l1', 'app_list'], axis=1, inplace=True)

#### Model

In [ ]:
df_customer_mapped_v4_age_w.head(1)

In [ ]:
evaluate_target_vs_rest(df_customer_mapped_v4_age_w, 'age_group')

#### Feature selection 

## Multi-Classification 

In [ ]:
def multi_classification(df,target_column):

    # Splitting data into train and test sets
    X = df.drop(columns=[target_column])
    y = df[target_column]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Initialize and fit the classifier
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)

    # Make predictions
    y_pred = clf.predict(X_test)

    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='weighted')
    recall = recall_score(y_test, y_pred, average='weighted')
    f1 = f1_score(y_test, y_pred, average='weighted')

    # Get top features
    feature_importances = clf.feature_importances_
    top_features_indices = feature_importances.argsort()[::-1][:20]  # Adjust number of top features as needed
    top_feature_names = X.columns[top_features_indices]

    print("Accuracy:", accuracy)
    print("Precision:", precision)
    print("Recall:", recall)
    print("F1 Score:", f1)
    print("Top Feature Names:", top_feature_names)

In [ ]:
multi_classification(df_customer_mapped_v1_age_w, 'age_group')

### Next classify step 

In [ ]:
# Keep customer_id &  values for reference
customer_ids = df_customer_mapped_age_na['customer_id']

In [ ]:
# Step 2: Predicting with the Model
for col in df_customer_mapped_age_na.columns:
    df_customer_mapped_age_na[col] = label_encoder.fit_transform(df_customer_mapped_age_na[col])

    
# Predict age_group for df_customer_mapped_age_na
df_customer_mapped_age_na['predicted_age_group'] = model.predict(df_customer_mapped_age_na\
                                                                 .drop(columns=['customer_id','age','age_group']))
df_customer_mapped_age_na['age_group'] = df_customer_mapped_age_na['predicted_age_group']
df_customer_mapped_age_na = df_customer_mapped_age_na.drop('predicted_age_group', axis=1)

# Add customer_id column back 
df_customer_mapped_age_na['customer_id'] = customer_ids

In [ ]:
df_customer_mapped_age_w.columns

In [ ]:
df_customer_mapped_age_na.columns

In [ ]:
df_customer_mapped_age_w.head(1)

In [ ]:
df_customer_mapped_age_na.head(1)

In [ ]:
df_customer_mapped_age_w.groupby(['age_group']).agg(customer = ('customer_id','nunique'))

In [ ]:
df_customer_mapped_age_na.groupby(['age_group']).agg(customer = ('customer_id','nunique'))

In [ ]:
df_classified_data = pd.concat([df_customer_mapped_age_w, df_customer_mapped_age_na])

# If you want to reset the index after concatenating
df_classified_data.reset_index(drop=True, inplace=True)
df_classified_data = df_classified_data[['customer_id', 'age_group']]


df_classified_data = pd.merge(df_customer_mapped[['customer_id', 'top_categories']], 
                              df_classified_data,
                              how='left',
                              on='customer_id')

df_classified_data.head(2)

In [ ]:
df_classified_data.age_group.unique()

In [ ]:
unique_categories2 = set(category for sublist in df_classified_data['top_categories'] for category in sublist)
unique_categories2 = list(unique_categories2)
print("Count of unique category:", len(unique_categories2))
sorted(unique_categories2)

In [ ]:
for category in unique_categories2:
    df_classified_data[category] = df_classified_data['top_categories'].apply(lambda x: 1 if category in x else 0) 
    
df_classified_data.drop('top_categories', axis=1, inplace=True)    
df_classified_data.head(1)

## Triangulation 2 - Customer use-case

In [ ]:
df_all_customer.columns

In [ ]:
df_all_customer[['customer_id', # 'morning_fe', 'evening_fe', 
                 'morning_rr', 'evening_rr',
                 'first_half_drop_primary', 'first_half_drop_secondary',
                 'second_half_pickup_primary', 'second_half_pickup_secondary']]

In [ ]:
df_triangulation_2 = df_all_customer[(~df_all_customer['first_half_drop_primary'].isna() 
                                      | 
                                      ~df_all_customer['first_half_drop_secondary'].isna()
                                      |
                                      ~df_all_customer['second_half_pickup_primary'].isna() 
                                      | 
                                      ~df_all_customer['second_half_pickup_secondary'].isna()
                                     )]\
                                    [['customer_id', 'age', 'morning_rr', 'evening_rr',
                                      'first_half_drop_primary','first_half_drop_secondary',
                                      'second_half_pickup_primary', 'second_half_pickup_secondary']]

df_triangulation_2.head(5)

In [ ]:
df_triangulation_2['primary_use_case'] = np.where(df_triangulation_2['morning_rr'] > df_triangulation_2['evening_rr'],
                                             df_triangulation_2['first_half_drop_primary'],
                                             np.where(df_triangulation_2['morning_rr'] < df_triangulation_2['evening_rr'],
                                                      df_triangulation_2['second_half_pickup_primary'],
                                                      df_triangulation_2['first_half_drop_primary']))
df_triangulation_2['secondary_use_case'] = np.where(df_triangulation_2['morning_rr'] > df_triangulation_2['evening_rr'],
                                                 df_triangulation_2['first_half_drop_secondary'],
                                                 np.where(df_triangulation_2['morning_rr'] < df_triangulation_2['evening_rr'],
                                                          df_triangulation_2['second_half_pickup_secondary'],
                                                          df_triangulation_2['first_half_drop_secondary']))

df_use_case_data = df_triangulation_2[['customer_id', 'primary_use_case', 'secondary_use_case']]
df_use_case_data

## Triangulation 3 - Customer metric

In [ ]:
df_all_customer.columns

In [ ]:
df_all_customer[['customer_id', 'fe_count', 'rr_count', 'net_count',
                 'ltr_segment', 'life_time_rides', 'life_time_stage', 'service_affinity']]

In [ ]:
df_triangulation_3 = df_all_customer[['customer_id', 'net_count', 'ltr_segment', 
                                      'life_time_rides', 'life_time_stage', 'service_affinity']]
df_triangulation_3.rename(columns = {'net_count' :'rpc_monthly'}, inplace = True)
df_metric_data = df_triangulation_3
df_metric_data

# Triangulation

In [ ]:
df_all_customer.head(1)

In [ ]:
df_classified_data.head(1)

In [ ]:
df_use_case_data.head(1)

In [ ]:
df_metric_data.head(1)

## Triangulation of 3 data sources

In [ ]:
df_consolidate=df_all_customer[df_all_customer['platform'].isin(['android'])][['platform', 'customer_id', 'age', 'gender']]
df_consolidate.head(1)

In [ ]:
df_consolidate.customer_id.nunique()

In [ ]:
df1_merge = pd.merge(df_consolidate,df_metric_data,how='left',on='customer_id')
df2_merge = pd.merge(df1_merge,df_use_case_data,how='left',on='customer_id')
df_classified_data.rename(columns = {'customer_id' :'user_id'}, inplace = True)
df3_merge = pd.merge(df2_merge,df_classified_data,how='left',left_on='customer_id', right_on = 'user_id')

df3_merge = df3_merge[['platform', 'customer_id', 'gender', 'age', 'age_group',
                       'rpc_monthly', 'ltr_segment','life_time_rides', 'life_time_stage', 'service_affinity',
                       'primary_use_case', 'secondary_use_case',  
                       'delivery_food', 'driver_app', 'core_office', 'office', 'core_education', 'gaming',
                       'entertainment', 'social', 'dating', 'finance_investment', 'travel_bookings',  
                       'streaming_music', 'travel_ridehailing', #'rest', 
                       'user_id']]
df3_merge

In [ ]:
df3_merge.customer_id.nunique()

### Derived values

In [ ]:
df3_merge['age_presence'] = np.where(df3_merge['age'] > 0, 'AVAILABLE', 'NA') 
df3_merge['use_case_presence'] = np.where((df3_merge['primary_use_case'].isna()) 
                                          & 
                                          (df3_merge['secondary_use_case'].isna()), 'NA', 'AVAILABLE')

df3_merge['appo_presence'] = np.where((df3_merge['user_id'].isna() == True),
                                      'NA', 'AVAILABLE')

In [ ]:
df3_merge.head(1)

In [ ]:
df3_merge.customer_id.nunique()

In [ ]:
df_temp = df_all_customer[df_all_customer['platform'] == 'android']\
                .groupby(['platform', 'age_presence'])\
                .agg(customers = ('customer_id', 'nunique'))\
                .reset_index()
df_temp['total'] = df_temp.groupby('platform')['customers'].transform('sum')
df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp['total'], 2)
df_temp

In [ ]:
df_temp = df3_merge\
                .groupby(['age_presence','use_case_presence','appo_presence'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df3_merge.customer_id.nunique(), 2)
df_temp

In [ ]:
df_temp['% distribution'].sum()

In [ ]:
df3_merge.head()

In [ ]:
df3_merge.customer_id.nunique()

### Confidence level

In [ ]:
df3_merge.age_group.unique()

In [ ]:
def confidence(row):
    
    if row['age_presence'] == 'AVAILABLE' :
        return 'Age Exist'
    

    elif row['appo_presence'] == 'NA' :        
        return 'Unknown' 
    
    
    elif row['use_case_presence'] == 'AVAILABLE' and row['appo_presence'] == 'AVAILABLE' :
        
        
        if row['core_office'] == 1 and row['core_education'] == 1 :
        
            if row['age_group'] in ['15-20', '21-25'] :
                return 'High'
            
            elif row['age_group'] in ['26-35'] :
                return 'Medium'
            
            elif row['age_group'] in ['36-60'] :
                return 'Low'
            
            
            else:
                return 'check'
            
        
        ##Office
        
        elif row['core_office'] == 1 and row['age_group'] in ['26-35', '36-60'] :
            
            if row['primary_use_case'] == 'office' or row['secondary_use_case'] == 'office' :
                return 'High'
        
            elif row['finance_investment'] == 1 :
                return 'High'
            
            elif row['office'] == 1 :
                return 'High'
            
            else:  
                return 'Medium'
            
        elif row['core_office'] == 1 and row['age_group'] in ['15-20', '21-25'] :
            
            if row['age_group'] in ['21-25'] :
                return 'Medium'
        
            else :
                return 'Low'


        ## Education 
        
        elif row['core_education'] == 1 :
            
            if row['primary_use_case'] == 'educational' or row['secondary_use_case'] == 'educational' :
                return 'High'
        
            elif row['dating'] == 1 :
                return 'High'

            elif row['age_group'] in ['15-20', '21-25'] :
                return 'High'
            
            elif row['age_group'] not in ['15-20', '21-25'] : 
                
                if row['primary_use_case'] in ['residential', 'office', 'food'] or row['primary_use_case'] in ['residential', 'office', 'food'] :
                    return 'High'
            
                else :
                    return 'Medium'
            
            else: 
                return 'Low'
                
        
    
        ## Dating 
        
        elif row['dating'] == 1 :
            
            if row['age_group'] in ['15-20', '21-25', '26-35'] :
                return 'High'
            
            elif row['finance_investment'] == 1 :
                return 'High'
        
            else: 
                return 'Low'
            
            
        ## Driver App 
        
        elif row['driver_app'] == 1 :
            
            if row['age_group'] in ['21-25', '26-35'] and row['gender'] == 'MALE' :
                return 'High'
            
            elif row['age_group'] in ['36-60'] :
                return 'Medium'
            
            elif row['gender'] == 'FEMALE' :
                return 'Low'
        
            else: 
                return 'Low'   
            
            
        ## Driver App 
        
        elif row['finance_investment'] == 1 :
            
            if row['age_group'] in ['26-35', '36-60'] and row['office'] == 1 :
                return 'High'
            
            elif row['age_group'] in ['26-35'] and (row['office'] == 1 or row['primary_use_case'] == 'educational' or row['secondary_use_case'] == 'educational'):
                return 'Medium'
            
            elif row['age_group'] in ['21-25'] or row['primary_use_case'] == 'office' or row['secondary_use_case'] == 'office' :
                return 'Low'
        
            else: 
                return 'Medium'   
        
        elif row['gaming'] == 1 :

            if row['age_group'] in ['15-20', '21-25', '26-35'] :
                return 'High'

            else: 
                return 'Low'   


        elif row['delivery_food'] == 1 and row['office'] == 1 and row['entertainment'] == 1 :

            if row['age_group'] in ['26-35'] :
                return 'High'

            else: 
                return 'Medium'        
            
        elif row['gender'] == 'FEMALE' :

            if row['age_group'] in ['21-25', '26-35', '36-60'] and row['service_affinity'] in ['ONLY_AUTO', 'ONLY_CAB', 'AUTO_CAB'] :
                return 'High'
            
            elif row['age_group'] in ['26-35', '36-60'] and row['service_affinity'] not in ['ONLY_LINK'] :
                return 'Low'

            else: 
                return 'Medium'    
            
        elif row['primary_use_case'] == 'office' or row['secondary_use_case'] == 'office' :

            if row['age_group'] in ['26-35', '36-60'] :
                return 'High'

            elif row['age_group'] in ['21-25'] :
                return 'Medium'

            else: 
                return 'Low'    

        elif row['ltr_segment'] == 'HH' :

            if row['primary_use_case'] != None :
                return 'Medium'

            elif row['secondary_use_case'] == None :
                return 'Low'
            
        elif row['delivery_food'] == 1 and row['travel_ridehailing'] == 1 :

            if row['age_group'] in ['26-35'] :
                return 'Medium'

            else :
                return 'Low'
            
        elif row['primary_use_case'] in ['educational'] :

            if row['age_group'] in ['15-20', '21-25'] :
                return 'Medium'

            else :
                return 'Low'

            
        else:
            return 'Unknown'

In [ ]:
df3_merge['confidence_tag'] = df3_merge.apply(confidence, axis=1)
df3_merge['confidence_tag'] = df3_merge['confidence_tag'].fillna('Low')

In [ ]:
df_temp = df3_merge\
                .groupby(['age_presence','use_case_presence','appo_presence'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df3_merge.customer_id.nunique(), 2)
df_temp

In [ ]:
df_temp['% distribution'].sum()

In [ ]:
df_temp = df3_merge\
                .groupby(['age_presence','use_case_presence','appo_presence', 'confidence_tag'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df3_merge.customer_id.nunique(), 2)
df_temp

In [ ]:
df_temp['% distribution'].sum()

In [ ]:
df3_merge[df3_merge['confidence_tag'].isna()]

In [ ]:
df_temp = df3_merge\
                .groupby(['age_presence', 'appo_presence', 'confidence_tag'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

In [ ]:
df_temp = df3_merge\
                .groupby(['age_presence', 'confidence_tag'])\
                .agg(customers = ('customer_id','nunique'))\
                .reset_index()

df_temp['% distribution'] = round(df_temp['customers']*100.00/df_temp.customers.sum(), 2)
df_temp

In [ ]:
df_temp['% distribution'].sum()

### Final Dataset

In [ ]:
df3_merge[['platform','customer_id','age','age_group']]